In [2]:
!pip install pandas numpy matplotlib seaborn plotly geopandas scikit-learn statsmodels jupyterlab streamlit duckdb

  Using cached duckdb-1.5.5-cp314-cp314-win_amd64.whl.metadata (4.2 kB)
  Using cached fqdn-1.5.1-py3-none-any.whl.metadata (1.4 kB)
  Using cached isoduration-20.11.0-py3-none-any.whl.metadata (5.7 kB)
  Using cached jsonpointer-3.1.1-py3-none-any.whl.metadata (2.4 kB)
  Using cached rfc3987_syntax-1.1.0-py3-none-any.whl.metadata (7.7 kB)
  Using cached uri_template-1.3.0-py3-none-any.whl.metadata (8.8 kB)
  Using cached webcolors-25.10.0-py3-none-any.whl.metadata (2.2 kB)
  Using cached lark-1.3.1-py3-none-any.whl.metadata (1.8 kB)
Using cached duckdb-1.5.5-cp314-cp314-win_amd64.whl (13.7 MB)
Using cached jsonpointer-3.1.1-py3-none-any.whl (7.7 kB)
Using cached rfc3987_syntax-1.1.0-py3-none-any.whl (8.0 kB)
Using cached lark-1.3.1-py3-none-any.whl (113 kB)
Using cached webcolors-25.10.0-py3-none-any.whl (14 kB)
Using cached fqdn-1.5.1-py3-none-any.whl (9.1 kB)
Using cached isoduration-20.11.0-py3-none-any.whl (11 kB)
Using cached uri_template-1.3.0-py3-none-any.whl (11 kB)

   ------

In [8]:
import pandas as pd
listings = pd.read_csv('../data/listings.csv.gz', compression='gzip', low_memory=False)
calendar = pd.read_csv('../data/calendar.csv.gz', compression='gzip')
reviews = pd.read_csv('../data/reviews.csv')
print(listings.shape, calendar.shape, reviews.shape)
listings.info()
listings.head(3)

(24876, 90) (9092881, 5) (1886932, 2)
<class 'pandas.DataFrame'>
RangeIndex: 24876 entries, 0 to 24875
Data columns (total 90 columns):
 #   Column                                        Non-Null Count  Dtype  
---  ------                                        --------------  -----  
 0   id                                            24876 non-null  int64  
 1   listing_url                                   24876 non-null  str    
 2   scrape_id                                     24876 non-null  int64  
 3   last_scraped                                  24876 non-null  str    
 4   source                                        24876 non-null  str    
 5   name                                          24876 non-null  str    
 6   description                                   24263 non-null  str    
 7   neighborhood_overview                         0 non-null      float64
 8   picture_url                                   24862 non-null  str    
 9   host_id                           

,id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,picture_url,host_id,...,review_scores_communication,review_scores_location,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
0,6499,https://www.airbnb.com/rooms/6499,20260623040800,2026-07-02,city scrape,Belém 1 Bedroom Historical Apartment,"This apartment is all about Location, next to ...",NaN,https://a0.muscache.com/pictures/hosting/Hosti...,14455,...,4.88,4.81,4.37,NaN,NaN,1,1,0,0,0.83
1,25659,https://www.airbnb.com/rooms/25659,20260623040800,2026-06-26,city scrape,Heart of Lisbon Alfama. Flexible check in times,Cozy apartment in Lisbon's most historic and b...,NaN,https://a0.muscache.com/pictures/miso/Hosting-...,107347,...,4.95,4.86,4.81,56539/AL.,NaN,1,1,0,0,1.57
2,29396,https://www.airbnb.com/rooms/29396,20260623040800,2026-06-25,city scrape,Alfama Hill - Boutique apartment,Feel at home in the historic centre of Lisbon.,NaN,https://a0.muscache.com/pictures/163913/7d622c...,126415,...,4.91,4.87,4.73,28737/AL,NaN,1,1,0,0,2.59


In [9]:
### 📝 Data Acquisition & Quality Check Summary

#### 1. Raw Dataset Shapes
* **Listings**: (24,876 rows, 90 columns)
* **Calendar**: (9,092,881 rows, 5 columns)
* **Reviews**: (1,886,932 rows, 2 columns)

#### 2. Snapshot Price Status (calendar.csv)
* **Status**: **YES** – The `calendar.csv` file **does** include a `price` column in this specific snapshot layout (which contributes to its massive 9 million+ row scale). 
* **Note**: In `listings.csv.gz`, the column `price` is parsed as a text string object (e.g., `"$1,250.00"` or containing formatting symbols) and contains missing data (`22,636` non-null vs `24,876` total records). It will require regex cleaning to convert into numerical floats before OLS modeling.

#### 3. Initial Observations: 'Things that look messy'
* **Entirely Empty Columns (100% Missing)**: Columns like `neighborhood_overview`, `host_since`, `host_response_time`, `host_response_rate`, `host_acceptance_rate`, `host_thumbnail_url`, `host_neighbourhood`, `host_total_listings_count`, `host_verifications`, `neighbourhood`, `calendar_updated`, and `instant_bookable` have `0` non-null values. They are completely blank and must be dropped.
* **Corrupted Numeric Column**: The `price` field is mapped as a `str` (text object) instead of a `float` due to formatting flags like currency symbols (`$`) and comma thousands separators.
* **Structural Discrepancy**: While structural details like `bathrooms_text` have almost full completeness (24,862 values), the standalone numeric `bathrooms` column has thousands of missing entries (21,161 values), implying data parsing fragmentation.
* **Review Metric Missingness**: Over 3,100 listings lack historical reviews entirely, creating empty clusters (`NaN`) in fields like `review_scores_rating`, `review_scores_accuracy`, and `reviews_per_month`.


SyntaxError: invalid syntax (912284168.py, line 4)